# FraudLens - Production ML Pipeline for Transaction Fraud Detection

**Dataset:** IEEE-CIS Fraud Detection (Kaggle) - 590K e-commerce transactions, 434 merged features, ~3.5% fraud

**Stack:** pandas - XGBoost - LightGBM - SHAP - MLflow - FastAPI - Evidently - Docker

---

## How this notebook relates to the rest of the repo

The transforms live in the `fraudlens/` package rather than in these cells, and
the notebook imports them. That is deliberate: the API imports the *same*
objects, so a feature can never be computed one way in training and another way
at serving time. `scripts/train.py` runs this pipeline headlessly; this notebook
is the annotated walkthrough of what it does and why.

## Structure

| Section | Topic |
|---|---|
| 0 | Setup and data loading |
| 1 | EDA - imbalance, missingness, temporal structure |
| 2 | Temporal split, then leak-free feature engineering |
| 3 | Preprocessing with a frozen column contract |
| 4 | Model training - logistic baseline, XGBoost, LightGBM |
| 5 | MLflow experiment tracking |
| 6 | Evaluation on the sealed test slice |
| 7 | Calibration and cost-based threshold selection |
| 8 | SHAP explainability |
| 9 | Artifact export for serving |
| 10 | Drift monitoring |
| 11 | Summary |

## Section 0 - Setup and data loading

In [ ]:
# Dependencies live in requirements.txt: pip install -r requirements.txt
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

# Make the fraudlens package importable when running from the repo root.
ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import joblib
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from fraudlens.config import (
    DATA_DIR, MODEL_DIR, PLOTS_DIR, REPORTS_DIR,
    TARGET, TIME_COL, TRAIN_END_Q, VAL_END_Q, ensure_dirs,
)

# Create every output directory up front, before any cell writes a figure.
ensure_dirs()

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "#f8f9fa",
    "axes.grid": True,
    "grid.alpha": 0.35,
    "axes.spines.top": False,
    "axes.spines.right": False,
})
FRAUD, LEGIT, ACCENT = "#e74c3c", "#2ecc71", "#2c3e50"

print("Imports ready")

In [ ]:
# The two source tables are joined on TransactionID.
#
# Kaggle ships test_identity.csv with hyphenated column names (id-01) while
# train_identity.csv uses underscores (id_01). load_split normalises them;
# without that, the test frame silently loses ~20 identity features and the
# fitted imputer then rejects it.
from fraudlens.data import load_split

if not (DATA_DIR / "train_transaction.csv").exists():
    print("No data found. Choose one:")
    print("  python scripts/make_synthetic_data.py   # runnable stand-in, no Kaggle account")
    print("  python scripts/download_data.py         # the real competition data")
    raise SystemExit

raw = load_split("train")

print(f"Merged shape : {raw.shape[0]:,} rows x {raw.shape[1]} columns")
print(f"Fraud rate   : {raw[TARGET].mean():.3%} "
      f"({raw[TARGET].sum():,} fraud / {len(raw):,} total)")
print(f"Imbalance    : {(raw[TARGET] == 0).sum() / max((raw[TARGET] == 1).sum(), 1):.1f} : 1")

## Section 1 - Exploratory Data Analysis

Three things drive every later decision: the class imbalance rules out accuracy
as a metric, the missingness pattern determines which features survive, and the
temporal structure dictates how we split.

In [ ]:
# 1.1 Class imbalance and amount distribution
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

counts = raw[TARGET].value_counts().sort_index()
axes[0].bar(["Legit", "Fraud"], counts.to_numpy(), color=[LEGIT, FRAUD], edgecolor="white")
axes[0].set_title("Class distribution", fontweight="bold")
axes[0].set_ylabel("Transactions")
for i, v in enumerate(counts.to_numpy()):
    axes[0].text(i, v, f"{v:,}\n({v / len(raw) * 100:.1f}%)",
                 ha="center", va="bottom", fontsize=9, fontweight="bold")

# Plot the two classes as separate series rather than via groupby/unstack, which
# would build a 590,000-column intermediate frame.
for label, colour, name in [(0, LEGIT, "Legit"), (1, FRAUD, "Fraud")]:
    axes[1].hist(np.log1p(raw.loc[raw[TARGET] == label, "TransactionAmt"]),
                 bins=60, alpha=0.6, color=colour, label=name, density=True)
axes[1].set_title("log(1 + TransactionAmt) by class", fontweight="bold")
axes[1].set_xlabel("log(1 + amount)")
axes[1].set_ylabel("Density")
axes[1].legend()

plt.tight_layout()
plt.savefig(PLOTS_DIR / "01_class_distribution.png", dpi=140, bbox_inches="tight")
plt.show()

print(f"At {raw[TARGET].mean():.2%} fraud, a model predicting 'never fraud' scores "
      f"{1 - raw[TARGET].mean():.2%} accuracy. Accuracy is useless here; "
      f"we optimise PR-AUC.")

In [ ]:
# 1.2 Missingness
missing = (raw.isnull().mean() * 100).sort_values(ascending=False)

print(f"Columns >80% missing: {(missing > 80).sum()}  (these get dropped)")
print(f"Columns >50% missing: {(missing > 50).sum()}")
print(f"Columns with no missing values: {(missing == 0).sum()}")

fig, ax = plt.subplots(figsize=(13, 5))
top = missing.head(50)
ax.barh(range(len(top)), top.to_numpy(),
        color=[FRAUD if v > 80 else "#f39c12" if v > 50 else LEGIT for v in top])
ax.set_yticks(range(len(top)))
ax.set_yticklabels(top.index, fontsize=7)
ax.invert_yaxis()
ax.axvline(80, color="red", ls="--", alpha=0.7, label="80% drop threshold")
ax.set_xlabel("Missing %")
ax.set_title("Top 50 columns by missingness", fontweight="bold")
ax.legend()
plt.tight_layout()
plt.savefig(PLOTS_DIR / "02_missingness.png", dpi=140, bbox_inches="tight")
plt.show()

In [ ]:
# 1.3 Temporal structure - this is what forces a time-based split
raw["_abs_day"] = raw[TIME_COL] // 86400
by_day = raw.groupby("_abs_day")[TARGET].agg(["mean", "count"]).reset_index()

fig, ax1 = plt.subplots(figsize=(13, 4))
ax2 = ax1.twinx()
ax1.fill_between(by_day["_abs_day"], by_day["mean"] * 100, color=FRAUD, alpha=0.35)
ax1.plot(by_day["_abs_day"], by_day["mean"] * 100, color=FRAUD, lw=1.4, label="Fraud rate %")
ax2.plot(by_day["_abs_day"], by_day["count"], color=ACCENT, lw=1.1, ls="--",
         alpha=0.75, label="Volume")

# Mark where the three-way split falls.
for q, name in [(TRAIN_END_Q, "train|val"), (VAL_END_Q, "val|test")]:
    boundary = raw[TIME_COL].quantile(q) // 86400
    ax1.axvline(boundary, color="black", ls=":", lw=1.5)
    ax1.text(boundary, ax1.get_ylim()[1] * 0.92, f" {name}", fontsize=8)

ax1.set_xlabel("Day (relative to dataset start)")
ax1.set_ylabel("Fraud rate (%)", color=FRAUD)
ax2.set_ylabel("Transaction volume", color=ACCENT)
ax1.set_title("Daily fraud rate and volume - fraud is non-stationary",
              fontweight="bold")
lines = ax1.get_legend_handles_labels()[0] + ax2.get_legend_handles_labels()[0]
labels = ax1.get_legend_handles_labels()[1] + ax2.get_legend_handles_labels()[1]
ax1.legend(lines, labels, loc="upper right", fontsize=8)
plt.tight_layout()
plt.savefig(PLOTS_DIR / "03_temporal_fraud.png", dpi=140, bbox_inches="tight")
plt.show()

raw.drop(columns="_abs_day", inplace=True)
print("Fraud rate moves over time, so a random split would let the model "
      "train on the same period it is scored on.")

In [ ]:
# 1.4 Fraud rate by categorical feature
present = [c for c in ["ProductCD", "card4", "card6", "DeviceType"] if c in raw.columns]
fig, axes = plt.subplots(1, len(present), figsize=(4 * len(present), 3.6))
axes = np.atleast_1d(axes)

for ax, col in zip(axes, present):
    rates = raw.groupby(col)[TARGET].agg(["mean", "count"]).sort_values("mean", ascending=False)
    bars = ax.bar(rates.index.astype(str), rates["mean"] * 100,
                  color=[FRAUD if v > raw[TARGET].mean() else LEGIT for v in rates["mean"]])
    ax.axhline(raw[TARGET].mean() * 100, color=ACCENT, ls="--", lw=1, alpha=0.7)
    ax.set_title(f"Fraud rate by {col}", fontsize=11, fontweight="bold")
    ax.set_ylabel("Fraud rate (%)")
    ax.tick_params(axis="x", rotation=30, labelsize=8)
    for bar, n in zip(bars, rates["count"]):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
                f"n={n:,}", ha="center", va="bottom", fontsize=7, color="gray")

plt.suptitle("Fraud rate by category (dashed line = base rate)", fontweight="bold")
plt.tight_layout()
plt.savefig(PLOTS_DIR / "04_categorical_fraud_rates.png", dpi=140, bbox_inches="tight")
plt.show()

## Section 2 - Split first, then engineer features

Order matters more than the feature list. Aggregate features such as
"how many transactions has this card made" are computed with a `value_counts()`
over whatever frame you hand them. Compute those before splitting and the
training rows absorb statistics from the validation period.

There is a second, worse consequence. A single incoming API request has no
frame to count over, so those features simply cannot be recomputed at serving
time. `FeatureEngineer` solves both problems the same way: `fit` learns the
lookup tables from training rows only, and they ship with the model.

In [ ]:
from fraudlens.features import FeatureEngineer

# Three-way chronological split. The third slice is what allows early stopping
# and threshold tuning to use `val` while `test` stays completely sealed.
t = raw[TIME_COL]
train_mask = t <= t.quantile(TRAIN_END_Q)
val_mask   = (t > t.quantile(TRAIN_END_Q)) & (t <= t.quantile(VAL_END_Q))
test_mask  = t > t.quantile(VAL_END_Q)

print(f"train {train_mask.sum():>7,}  fraud {raw.loc[train_mask, TARGET].mean():.3%}")
print(f"val   {val_mask.sum():>7,}  fraud {raw.loc[val_mask, TARGET].mean():.3%}")
print(f"test  {test_mask.sum():>7,}  fraud {raw.loc[test_mask, TARGET].mean():.3%}")
assert not (train_mask & val_mask).any() and not (val_mask & test_mask).any()

In [ ]:
# Fit on the training slice only, then transform everything.
engineer = FeatureEngineer().fit(raw.loc[train_mask])
engineered = engineer.transform(raw)

added = [c for c in engineered.columns if c not in raw.columns]
print(f"{raw.shape[1]} raw columns -> {engineered.shape[1]} ({len(added)} engineered)\n")

groups = {
    "temporal":    [c for c in added if c in ("hour", "day", "is_night", "is_weekend")],
    "amount":      [c for c in added if c.startswith("TransactionAmt") or c == "is_round_amount"],
    "card agg":    [c for c in added if c.startswith("card")],
    "D relative":  [c for c in added if c.endswith("_norm")],
    "email/device":[c for c in added if "email" in c or "Device" in c],
    "C / V stats": [c for c in added if c.startswith(("C_", "V_"))],
}
for name, cols in groups.items():
    print(f"  {name:<13} {len(cols):>3}  {', '.join(cols[:4])}{' ...' if len(cols) > 4 else ''}")

In [ ]:
# The property that makes serving safe: one row alone equals that row in a batch.
sample_position = 5
batch_row  = engineered.iloc[[sample_position]]
single_row = engineer.transform(raw.iloc[[sample_position]])

pd.testing.assert_frame_equal(single_row, batch_row[single_row.columns], check_dtype=False)
print("Single-row and batch feature vectors are identical.")

# And an unseen card degrades sensibly instead of producing NaN.
novel = raw.iloc[[0]].copy()
novel["card1"] = 999_999
out = engineer.transform(novel)
print(f"Unseen card -> card1_freq={out['card1_freq'].iloc[0]:.0f}, "
      f"amt_ratio={out['card1_amt_ratio'].iloc[0]:.3f} (falls back to the population mean)")

## Section 3 - Preprocessing with a frozen column contract

The numeric/categorical split is decided once, during `fit`, and stored. Deriving
it from dtypes at serving time is unsafe: label-encoded categoricals look
numeric, and any column the caller omits arrives as a float NaN. That mismatch
is what made every `/predict` call fail with an imputer shape error.

In [ ]:
from fraudlens.preprocessing import Preprocessor

preprocessor = Preprocessor().fit(engineered.loc[train_mask])
X = preprocessor.transform(engineered)
y = raw[TARGET].astype(int)

print(f"Dropped {len(preprocessor.dropped_cols_)} columns above the 80% missingness cap")
print(f"Model matrix: {X.shape[1]} features "
      f"({len(preprocessor.num_cols_)} numeric, {len(preprocessor.cat_cols_)} categorical)")
print(f"Remaining nulls: {int(X.isnull().sum().sum())}")

X_train, y_train = X.loc[train_mask], y.loc[train_mask]
X_val,   y_val   = X.loc[val_mask],   y.loc[val_mask]
X_test,  y_test  = X.loc[test_mask],  y.loc[test_mask]

In [ ]:
# A five-field API payload must produce the exact same 361-column matrix.
payload = pd.DataFrame([{
    "TransactionAmt": 299.99, "ProductCD": "W", "card4": "visa",
    "P_emaildomain": "gmail.com", "TransactionDT": 86400.0,
}])
served = preprocessor.transform(engineer.transform(payload))

print(f"Sparse request -> {served.shape[1]} features, "
      f"column order identical: {list(served.columns) == list(X.columns)}")
print(f"Nulls after imputation: {int(served.isnull().sum().sum())}")

## Section 4 - Model training

A logistic baseline establishes what the gradient-boosted models have to beat.
All three see class imbalance explicitly: `class_weight="balanced"` for the
linear model, `scale_pos_weight` for the trees.

Early stopping watches the **validation** slice. The test slice is not touched
anywhere in this section.

In [ ]:
import time
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
import xgboost as xgb
import lightgbm as lgb

from fraudlens.modeling import ScaledModel, evaluate

scale_pos_weight = (y_train == 0).sum() / max((y_train == 1).sum(), 1)
print(f"scale_pos_weight = {scale_pos_weight:.1f}")

models, timings = {}, {}

# Baseline. lbfgs rather than saga: saga on 350+ columns took hours here
# without beating lbfgs.
scaler = StandardScaler()
t0 = time.time()
lr = LogisticRegression(class_weight="balanced", max_iter=1000,
                        solver="lbfgs", C=0.1, random_state=42, n_jobs=-1)
lr.fit(scaler.fit_transform(X_train), y_train)
timings["Logistic Regression"] = time.time() - t0
# ScaledModel is defined in fraudlens.modeling, not here, so the artifact can be
# unpickled by the API. A class defined in a notebook lives in __main__ and
# cannot be loaded by another process.
models["Logistic Regression"] = ScaledModel(lr, scaler)
print(f"Logistic Regression trained in {timings['Logistic Regression']:.1f}s")

In [ ]:
t0 = time.time()
xgb_model = xgb.XGBClassifier(
    n_estimators=800, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    eval_metric="aucpr",           # PR, not ROC - the positive class is what matters
    early_stopping_rounds=50,
    tree_method="hist", random_state=42, n_jobs=-1,
)
xgb_model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
timings["XGBoost"] = time.time() - t0
models["XGBoost"] = xgb_model
print(f"XGBoost stopped at iteration {xgb_model.best_iteration} "
      f"in {timings['XGBoost']:.1f}s")

In [ ]:
t0 = time.time()
lgb_model = lgb.LGBMClassifier(
    n_estimators=1500, num_leaves=63, learning_rate=0.05,
    subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
    min_child_samples=20, scale_pos_weight=scale_pos_weight,
    random_state=42, n_jobs=-1, verbose=-1,
)
lgb_model.fit(X_train, y_train, eval_set=[(X_val, y_val)],
              eval_metric="average_precision",
              callbacks=[lgb.early_stopping(50, verbose=False)])
timings["LightGBM"] = time.time() - t0
models["LightGBM"] = lgb_model
print(f"LightGBM stopped at iteration {lgb_model.best_iteration_} "
      f"in {timings['LightGBM']:.1f}s")

In [ ]:
# Compare on validation, and select on PR-AUC.
val_results, val_probas = {}, {}
for name, model in models.items():
    proba = model.predict_proba(X_val)[:, 1]
    metrics = evaluate(y_val, proba)
    metrics["train_time_s"] = round(timings[name], 1)
    val_results[name], val_probas[name] = metrics, proba

results_df = pd.DataFrame(val_results).T
print(results_df[["roc_auc", "pr_auc", "brier", "f1", "precision", "recall"]].round(4).to_string())

# ROC-AUC saturates when 96.5% of rows are negative; PR-AUC separates
# candidates far more sharply and reflects performance on the class we care about.
best_name = results_df["pr_auc"].idxmax()
best_model = models[best_name]
print(f"\nSelected on PR-AUC: {best_name}")

## Section 5 - MLflow experiment tracking

The tracking URI comes from `fraudlens.config`, so this notebook, `make mlflow`
and the Docker Compose service all read and write the same store. If they
diverge, the UI silently shows an empty experiment list.

In [ ]:
from fraudlens.config import MLFLOW_EXPERIMENT, MLFLOW_TRACKING_URI

run_ids = {}
try:
    import mlflow

    mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
    mlflow.set_experiment(MLFLOW_EXPERIMENT)

    params = {
        "Logistic Regression": {"solver": "lbfgs", "C": 0.1, "class_weight": "balanced"},
        "XGBoost": {"max_depth": 6, "learning_rate": 0.05, "scale_pos_weight": scale_pos_weight},
        "LightGBM": {"num_leaves": 63, "learning_rate": 0.05, "scale_pos_weight": scale_pos_weight},
    }

    for name, metrics in val_results.items():
        with mlflow.start_run(run_name=name) as run:
            mlflow.log_params(params[name])
            mlflow.log_params({
                "train_size": len(X_train), "val_size": len(X_val),
                "test_size": len(X_test), "feature_count": X.shape[1],
            })
            mlflow.log_metrics({k: v for k, v in metrics.items() if isinstance(v, (int, float))})
            mlflow.set_tag("selected", str(name == best_name))
            run_ids[name] = run.info.run_id
            print(f"  logged {name}  run {run.info.run_id[:8]}")

    print(f"\nmlflow ui --port 5000 --backend-store-uri {MLFLOW_TRACKING_URI}")
except ImportError:
    print("mlflow not installed; skipping (pip install mlflow)")
except Exception as exc:
    print(f"mlflow unavailable: {exc}")

## Section 6 - Evaluation

In [ ]:
from sklearn.metrics import (
    roc_curve, precision_recall_curve, roc_auc_score,
    average_precision_score, confusion_matrix, classification_report,
)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for (name, proba), colour in zip(val_probas.items(), ["#3498db", "#e74c3c", "#2ecc71"]):
    fpr, tpr, _ = roc_curve(y_val, proba)
    axes[0].plot(fpr, tpr, color=colour, lw=2,
                 label=f"{name} ({roc_auc_score(y_val, proba):.4f})")
    prec, rec, _ = precision_recall_curve(y_val, proba)
    axes[1].plot(rec, prec, color=colour, lw=2,
                 label=f"{name} ({average_precision_score(y_val, proba):.4f})")

axes[0].plot([0, 1], [0, 1], "k--", alpha=0.4, label="Random (0.5)")
axes[0].set_xlabel("False positive rate")
axes[0].set_ylabel("True positive rate")
axes[0].set_title("ROC curves - all look strong", fontweight="bold")
axes[0].legend(loc="lower right", fontsize=8)

axes[1].axhline(y_val.mean(), color="gray", ls="--", alpha=0.6,
                label=f"Random ({y_val.mean():.3f})")
axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Precision")
axes[1].set_title("PR curves - the honest view under imbalance", fontweight="bold")
axes[1].legend(loc="upper right", fontsize=8)

plt.tight_layout()
plt.savefig(PLOTS_DIR / "05_roc_pr_curves.png", dpi=140, bbox_inches="tight")
plt.show()

## Section 7 - Calibration and threshold selection

Two separate problems, often conflated.

**Calibration.** Training with `scale_pos_weight` deliberately distorts the
output scale, so raw `predict_proba` values are not probabilities. An isotonic
fit on held-out data corrects the scale. It is monotonic, so ROC-AUC and PR-AUC
are unchanged - only the Brier score moves.

**Threshold.** 0.5 is arbitrary. Maximising F1 is also arbitrary: it weights
precision and recall equally, which encodes no business reality. A missed fraud
costs the disputed amount; a false positive costs manual review plus the churn
risk of blocking a real customer. We minimise total expected cost and report
what the F1 choice would have been for comparison.

In [ ]:
from fraudlens.modeling import CalibratedModel, sweep_thresholds, pick_threshold

calibrated = CalibratedModel.fit(best_model, X_val, y_val)
val_proba_cal = calibrated.predict_proba(X_val)[:, 1]

before, after = evaluate(y_val, val_probas[best_name]), evaluate(y_val, val_proba_cal)
print(f"Brier   {before['brier']:.4f} -> {after['brier']:.4f}   (lower is better)")
print(f"ROC-AUC {before['roc_auc']:.4f} -> {after['roc_auc']:.4f}  (unchanged: isotonic is monotonic)")
print(f"PR-AUC  {before['pr_auc']:.4f} -> {after['pr_auc']:.4f}")

In [ ]:
# Charge each missed fraud at its actual transaction amount.
amounts_val = raw.loc[val_mask, "TransactionAmt"].to_numpy()
sweep = sweep_thresholds(y_val, val_proba_cal, amounts=amounts_val)

cost_choice = pick_threshold(sweep, strategy="min_cost")
f1_choice   = pick_threshold(sweep, strategy="max_f1")

print(f"min-cost threshold : {cost_choice.threshold:.3f}  "
      f"precision {cost_choice.precision:.3f}  recall {cost_choice.recall:.3f}  "
      f"flags {cost_choice.flag_rate:.2%}")
print(f"max-F1 threshold   : {f1_choice.threshold:.3f}  "
      f"precision {f1_choice.precision:.3f}  recall {f1_choice.recall:.3f}  "
      f"flags {f1_choice.flag_rate:.2%}")
print(f"\nThe two disagree, which is the point: F1 has no notion of what a "
      f"missed fraud actually costs.")

THRESHOLD = cost_choice.threshold

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(sweep["threshold"], sweep["precision"], color="#3498db", label="Precision")
axes[0].plot(sweep["threshold"], sweep["recall"], color="#e74c3c", label="Recall")
axes[0].plot(sweep["threshold"], sweep["f1"], color="#9b59b6", lw=2, label="F1")
axes[0].axvline(THRESHOLD, color="black", ls="--", label=f"Chosen {THRESHOLD:.3f}")
axes[0].axvline(0.5, color="gray", ls=":", alpha=0.7, label="Naive 0.5")
axes[0].set_xlabel("Threshold")
axes[0].set_ylabel("Score")
axes[0].set_title("Precision / recall trade-off", fontweight="bold")
axes[0].legend(fontsize=8)

axes[1].plot(sweep["threshold"], sweep["expected_cost"], color="#16a085", lw=2)
axes[1].axvline(THRESHOLD, color="black", ls="--", label=f"Min cost {THRESHOLD:.3f}")
axes[1].axvline(f1_choice.threshold, color="#9b59b6", ls=":", label=f"Max F1 {f1_choice.threshold:.3f}")
axes[1].set_xlabel("Threshold")
axes[1].set_ylabel("Expected cost")
axes[1].set_title("Expected business cost", fontweight="bold")
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.savefig(PLOTS_DIR / "07_threshold_tuning.png", dpi=140, bbox_inches="tight")
plt.show()

In [ ]:
# Only now do we touch the test slice, once, at the chosen operating point.
from fraudlens.modeling import precision_at_recall

test_proba = calibrated.predict_proba(X_test)[:, 1]
test_metrics = evaluate(y_test, test_proba, threshold=THRESHOLD)
test_metrics["precision_at_80_recall"] = precision_at_recall(y_test, test_proba, 0.8)

print("Held-out test performance")
for k, v in test_metrics.items():
    print(f"  {k:<24} {v:.4f}")

cm = confusion_matrix(y_test, (test_proba >= THRESHOLD).astype(int))
fig, ax = plt.subplots(figsize=(5.5, 4.5))
sns.heatmap(cm, annot=True, fmt=",d", cmap="Blues", ax=ax,
            xticklabels=["Allow", "Block"], yticklabels=["Legit", "Fraud"],
            linewidths=0.5, linecolor="white", cbar=False)
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
ax.set_title(f"Confusion matrix - {best_name} @ {THRESHOLD:.3f}", fontweight="bold")
plt.tight_layout()
plt.savefig(PLOTS_DIR / "06_confusion_matrix.png", dpi=140, bbox_inches="tight")
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f"\nCaught {tp:,} of {tp + fn:,} frauds; "
      f"blocked {fp:,} legitimate customers out of {tn + fp:,}.")
print(classification_report(y_test, (test_proba >= THRESHOLD).astype(int),
                            target_names=["Legit", "Fraud"], digits=4))

## Section 8 - SHAP explainability

A score a risk analyst cannot interrogate is a score they will not act on.

SHAP explains the *uncalibrated* tree model, since `TreeExplainer` needs the raw
booster. Calibration is monotonic, so the ranking and sign of each contribution
carry over to the calibrated probability unchanged.

In [ ]:
import shap

SHAP_SAMPLE = min(2000, len(X_val))
X_shap = X_val.sample(SHAP_SAMPLE, random_state=42)

explainer = shap.TreeExplainer(best_model)
shap_raw = explainer.shap_values(X_shap)

# Normalise across SHAP versions and model types: older builds return a list
# per class, newer ones a 3-D array.
shap_values = shap_raw[1] if isinstance(shap_raw, list) else np.asarray(shap_raw)
if shap_values.ndim == 3:
    shap_values = shap_values[:, :, 1]

print(f"SHAP values: {shap_values.shape}")

importance = (pd.Series(np.abs(shap_values).mean(axis=0), index=X_shap.columns)
                .sort_values(ascending=False))
TOP_FEATURES = importance.index.tolist()

print("\nTop 15 fraud signals by mean |SHAP|")
for i, (feature, value) in enumerate(importance.head(15).items(), 1):
    print(f"  {i:2d}. {feature:<34} {value:.4f}")

In [ ]:
plt.figure(figsize=(10, 7))
shap.summary_plot(shap_values, X_shap, max_display=20, show=False, plot_type="dot")
plt.title(f"SHAP summary - {best_name}", fontweight="bold", pad=14)
plt.tight_layout()
plt.savefig(PLOTS_DIR / "08_shap_summary.png", dpi=140, bbox_inches="tight")
plt.show()

plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_shap, max_display=20, show=False, plot_type="bar")
plt.title(f"Mean |SHAP| - {best_name}", fontweight="bold")
plt.tight_layout()
plt.savefig(PLOTS_DIR / "09_shap_bar.png", dpi=140, bbox_inches="tight")
plt.show()

In [ ]:
# Explain the highest-risk transaction in the sample.
sample_proba = calibrated.predict_proba(X_shap)[:, 1]
position = int(np.argmax(sample_proba))

# X_shap was drawn with .sample(), so its index is a shuffled subset of y_val's.
# `position` is a positional offset into X_shap; resolving it back through
# X_shap.index is required. Using y_val.iloc[position] would report an
# unrelated row's label.
row_label = X_shap.index[position]
actual = y_val.loc[row_label]

print(f"Sample position   : {position}")
print(f"Row index in val  : {row_label}")
print(f"Fraud probability : {sample_proba[position]:.4f}")
print(f"Actual label      : {actual}  ({'FRAUD' if actual == 1 else 'legit'})")

base_value = explainer.expected_value
if isinstance(base_value, (list, np.ndarray)) and np.ndim(base_value) > 0:
    base_value = base_value[1] if len(base_value) > 1 else base_value[0]

explanation = shap.Explanation(
    values=shap_values[position],
    base_values=float(base_value),
    data=X_shap.iloc[position].to_numpy(),
    feature_names=X_shap.columns.tolist(),
)

plt.figure(figsize=(10, 7))
shap.waterfall_plot(explanation, max_display=15, show=False)
plt.title(f"Why this transaction scored {sample_proba[position]:.3f}",
          fontweight="bold", fontsize=11)
plt.tight_layout()
plt.savefig(PLOTS_DIR / "11_shap_waterfall.png", dpi=140, bbox_inches="tight")
plt.show()

## Section 9 - Export for serving

Four artifacts, and the API loads all of them. The feature engineer and
preprocessor are exported as fitted objects rather than being reimplemented in
the API - that duplication is exactly what caused training/serving skew.

In [ ]:
from fraudlens import drift
from fraudlens.config import MONITOR_ALWAYS

joblib.dump(engineer,     MODEL_DIR / "feature_engineer.joblib")
joblib.dump(preprocessor, MODEL_DIR / "preprocessor.joblib")
joblib.dump(calibrated,   MODEL_DIR / "model.joblib")      # calibrated, for scoring
joblib.dump(best_model,   MODEL_DIR / "base_model.joblib") # raw booster, for SHAP

# SHAP ranking alone leaves amount and time unmonitored, which is where a
# broken upstream feed shows up first, so pin those in regardless of rank.
monitored = list(dict.fromkeys([*MONITOR_ALWAYS, *TOP_FEATURES[:20]]))
reference = drift.build_reference(X_train, monitored)
drift.save_reference(reference, MODEL_DIR / "drift_reference.json")
monitored = list(reference)   # only these can ever report PSI

meta = {
    "model_name": best_name,
    "calibrated": True,
    "threshold_strategy": "min_cost",
    "optimal_threshold": round(THRESHOLD, 4),
    "f1_optimal_threshold": round(f1_choice.threshold, 4),
    "validation": {k: round(float(v), 4) for k, v in val_results[best_name].items()},
    "test": {k: round(float(v), 4) for k, v in test_metrics.items()},
    "roc_auc": round(test_metrics["roc_auc"], 4),
    "pr_auc": round(test_metrics["pr_auc"], 4),
    "f1": round(test_metrics["f1"], 4),
    "brier": round(test_metrics["brier"], 4),
    "flag_rate": round(test_metrics["flag_rate"], 4),
    "train_size": int(train_mask.sum()),
    "val_size": int(val_mask.sum()),
    "test_size": int(test_mask.sum()),
    "feature_count": int(X.shape[1]),
    "fraud_rate_train": round(float(y_train.mean()), 4),
    "top_features": TOP_FEATURES[:10],
    "monitored_features": monitored,
    "all_model_results": {k: {m: round(float(x), 4) for m, x in v.items()}
                          for k, v in val_results.items()},
    "mlflow_run_id": run_ids.get(best_name, ""),
}
(MODEL_DIR / "model_meta.json").write_text(json.dumps(meta, indent=2), encoding="utf-8")
sweep.to_csv(MODEL_DIR / "threshold_sweep.csv", index=False)

for f in sorted(MODEL_DIR.iterdir()):
    print(f"  {f.name:<28} {f.stat().st_size / 1024:>8.0f} KB")

In [ ]:
# Prove the exported artifacts reproduce this notebook's predictions.
reloaded_engineer = joblib.load(MODEL_DIR / "feature_engineer.joblib")
reloaded_prep     = joblib.load(MODEL_DIR / "preprocessor.joblib")
reloaded_model    = joblib.load(MODEL_DIR / "model.joblib")

check = raw.loc[test_mask].head(200)
round_trip = reloaded_model.predict_proba(
    reloaded_prep.transform(reloaded_engineer.transform(check))
)[:, 1]

assert np.allclose(round_trip, test_proba[:200], atol=1e-9)
print("Reloaded artifacts reproduce the in-notebook predictions exactly.")
print("\nStart the API:  python -m uvicorn api.main:app --reload --port 8000")

## Section 10 - Drift monitoring

Two layers. Evidently produces the rich offline HTML report for periodic review.
`fraudlens.drift` computes PSI from a small frozen reference and is what the
running API serves at `GET /drift`, because Evidently is too heavy for a request
path and its report object does not serialise into an API response.

In [ ]:
# Simulate a production window by shifting a few key features.
reference_data = X_train.sample(min(5000, len(X_train)), random_state=42).copy()
production_data = X_test.sample(min(2000, len(X_test)), random_state=99).copy()

# Only the top-20 features are monitored, so inject the shift into features
# that are actually watched. Drifting an unmonitored column would - correctly
# but unhelpfully - show up as no drift at all.
monitored_cols = [c for c in TOP_FEATURES[:20] if c in reference_data.columns]
drifted_features = monitored_cols[:3]
for feature in drifted_features:
    production_data[feature] = production_data[feature] + production_data[feature].std() * 0.8

print(f"Reference : {reference_data.shape}")
print(f"Production: {production_data.shape}")
print(f"Monitoring {len(monitored_cols)} features")
print(f"Injected an 0.8 std shift into: {drifted_features}")

In [ ]:
# Built-in PSI - the same code path the API uses.
reference_spec = drift.build_reference(reference_data, monitored_cols)
psi_report = drift.compute_drift(production_data, reference_spec)

print(f"Monitored {psi_report['features_monitored']} features")
print(f"Drifted   {psi_report['features_drifted']} "
      f"({psi_report['share_drifted']:.1%})   alert={psi_report['alert']}")
print("\nHighest PSI:")
for feature, info in list(psi_report["features"].items())[:10]:
    print(f"  {feature:<34} PSI={info['psi']:.4f}  {info['severity']}")

In [ ]:
# Evidently's HTML report. The 0.4.x and 0.6+ APIs differ substantially, so
# detect which is installed rather than pinning one and breaking on the other.
#
# The guard catches Exception, not just ImportError: evidently pulls in
# pydantic v1 shims that raise ConfigError at import time on Python 3.14.
# A drift report is a nice-to-have, so it must never break the notebook.
report_path = REPORTS_DIR / "drift_report.html"
evidently_api = None

try:
    from evidently.report import Report
    from evidently.metric_preset import DataDriftPreset
    evidently_api = "0.4"
except Exception:
    try:
        from evidently import Report
        from evidently.presets import DataDriftPreset
        evidently_api = "0.6+"
    except Exception as exc:
        print(f"evidently unavailable ({type(exc).__name__}); "
              f"the built-in PSI report above still applies.")

try:
    if evidently_api == "0.4":
        report = Report(metrics=[DataDriftPreset()])
        report.run(reference_data=reference_data[monitored_cols],
                   current_data=production_data[monitored_cols])
        report.save_html(str(report_path))
        summary = report.as_dict()["metrics"][0]["result"]
        print(f"Evidently 0.4.x: {summary['number_of_drifted_columns']}"
              f"/{summary['number_of_columns']} columns drifted")
    elif evidently_api == "0.6+":
        report = Report([DataDriftPreset()])
        result = report.run(current_data=production_data[monitored_cols],
                            reference_data=reference_data[monitored_cols])
        result.save_html(str(report_path))
        print("Evidently 0.6+ report written")
except Exception as exc:
    print(f"Evidently report failed ({type(exc).__name__}: {exc})")

if report_path.exists():
    print(f"Saved -> {report_path}")

## Section 11 - Summary

In [ ]:
summary = json.loads((MODEL_DIR / "model_meta.json").read_text(encoding="utf-8"))
baseline_pr = val_results["Logistic Regression"]["pr_auc"]

print(f'''
FraudLens - final numbers
{"=" * 68}

Data
  Rows                  {len(raw):,}  ({raw.shape[1]} merged columns)
  Fraud rate            {raw[TARGET].mean():.3%}
  Split (time-based)    train {summary["train_size"]:,} | val {summary["val_size"]:,} | test {summary["test_size"]:,}

Features
  Engineered            {len(added)} added on top of {raw.shape[1]} raw
  Model matrix          {summary["feature_count"]}
  Aggregates            frozen at fit time, so serving matches training exactly

Model: {summary["model_name"]} (isotonic-calibrated)
  Test ROC-AUC          {summary["test"]["roc_auc"]:.4f}
  Test PR-AUC           {summary["test"]["pr_auc"]:.4f}   (baseline LR {baseline_pr:.4f})
  Test Brier            {summary["test"]["brier"]:.4f}
  Precision / recall    {summary["test"]["precision"]:.4f} / {summary["test"]["recall"]:.4f}
  Flag rate             {summary["test"]["flag_rate"]:.2%}

Threshold
  Strategy              {summary["threshold_strategy"]}
  Chosen                {summary["optimal_threshold"]}  (F1-optimal would be {summary["f1_optimal_threshold"]})

Top signals
  {", ".join(summary["top_features"][:5])}

Monitoring
  {psi_report["features_drifted"]}/{psi_report["features_monitored"]} features drifted in the simulated window
{"=" * 68}

Every figure above comes from the sealed test slice, which was never used for
early stopping, model selection or threshold tuning.
''')